In [4]:

!pip -q install transformers


In [5]:

import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW

# 1. Load the BERT tokenizer and model
MODEL_NAME = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Device:", device)

# 2. Create a small sentiment dataset
texts = [
    "This movie is excellent",
    "I really enjoyed this film",
    "The product is very good",
    "Amazing experience",
    "I love this product",
    "The service was wonderful",
    "This movie is terrible",
    "I hated this film",
    "The product is very bad",
    "Very disappointing experience",
    "I dislike this product",
    "The service was awful"
]

# 1 = Positive, 0 = Negative
labels = [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]

# 3. Tokenize the text
encoding = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

input_ids = encoding["input_ids"]
attention_mask = encoding["attention_mask"]
labels = torch.tensor(labels)

# 4. Prepare the training dataset
dataset = TensorDataset(input_ids, attention_mask, labels)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True
)

# 5. Fine-tune BERT
optimizer = AdamW(model.parameters(), lr=2e-5)

epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in loader:
        ids, masks, batch_labels = [
            item.to(device) for item in batch
        ]

        optimizer.zero_grad()

        outputs = model(
            input_ids=ids,
            attention_mask=masks,
            labels=batch_labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(loader)

    print(
        f"Epoch {epoch + 1}/{epochs}, "
        f"Loss: {average_loss:.4f}"
    )

# 6. Predict sentiment for new text
def predict_sentiment(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=64
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)
        prediction = torch.argmax(
            outputs.logits, dim=1
        ).item()

    return "Positive" if prediction == 1 else "Negative"

# 7. Test the fine-tuned model
test_texts = [
    "The movie was fantastic",
    "This product is horrible",
    "I really love this experience"
]

print("\nSentiment Predictions:")

for text in test_texts:
    print(text, "->", predict_sentiment(text))

# 8. Save the fine-tuned model
model.save_pretrained("./fine_tuned_bert")
tokenizer.save_pretrained("./fine_tuned_bert")

print("\nModel saved successfully!")


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cpu
Epoch 1/3, Loss: 0.7201
Epoch 2/3, Loss: 0.7104
Epoch 3/3, Loss: 0.6564

Sentiment Predictions:
The movie was fantastic -> Positive
This product is horrible -> Positive
I really love this experience -> Positive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved successfully!
